In [2]:
!pip install -q faiss-cpu sentence-transformers openai google-generativeai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 85.9 MB/s eta 0:00:00


In [6]:
# ================================================================
# RAG on Academic Papers Dataset (AI/ML Research Papers 2023-2026)
# ================================================================

import os
import time
import pickle
import textwrap
import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer
import google.generativeai as genai
from openai import OpenAI
import io

# ---------- Check if running in Colab ----------
try:
    from google.colab import files, userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# ---------- Install required packages ----------
!pip install -q sentence-transformers faiss-cpu openai google-generativeai

# ================================================================
# 1. GET THE CSV FILE (upload or use embedded sample)
# ================================================================

print("📤 Upload your CSV file (or press Enter to use the embedded sample)")
try:
    uploaded = files.upload()
    if uploaded:
        filename = list(uploaded.keys())[0]
        content = uploaded[filename].decode('utf-8-sig')
        df = pd.read_csv(io.StringIO(content))
        print(f"✅ Loaded {len(df)} rows from {filename}")
    else:
        raise Exception("No file uploaded")
except:
    print("⚠️ No file uploaded. Using the embedded dataset (first 100 rows for demo).")
    # We'll use a small sample (you can replace this with the full dataset if needed)
    sample_data = {
        'title': ['MizAR 60 for Mizar 50', 'A Multi-Modal Distributed Real-Time IoT System', 'On a Method to Measure Supervised Multiclass Model’s Interpretability'],
        'doi': ['https://doi.org/10.4230/lipics.itp.2023.19', 'https://doi.org/10.4230/oasics.ng-res.2024.2', 'https://doi.org/10.4230/oasics.dx.2024.27'],
        'publication_date': ['2023-01-01', '2024-01-01', '2024-01-01'],
        'cited_by_count': [74707, 14276, 13164],
        'authors': ['Jakubův, Jan, Chvalovský, Karel, Goertzel, Zarathustra, Kaliszyk, Cezary, Olšák, Mirek', 'Khanam, Zeba, Achari, Vejey Pradeep Suresh, Boukhennoufa, Issam, Jindal, Anish, Singh, Amit Kumar', 'Gauriat, Charles-Maxime, Pencolé, Yannick, Ribot, Pauline, Brouillet, Gregory'],
        'journal': ['DROPS (Schloss Dagstuhl – Leibniz Center for Informatics)', 'DROPS (Schloss Dagstuhl – Leibniz Center for Informatics)', 'Dagstuhl Research Online Publication Server'],
        'open_access': [True, True, True],
        'type': ['preprint', 'preprint', 'preprint'],
        'topic': ['Machine Learning', 'Machine Learning', 'Machine Learning']
    }
    df = pd.DataFrame(sample_data)
    print(f"✅ Using sample of {len(df)} papers")

df.columns = df.columns.str.strip()
print("📋 Columns:", list(df.columns))

# ================================================================
# 2. CREATE CHUNKS (one per paper)
# ================================================================

chunks = []
for _, row in df.iterrows():
    chunk = (
        f"Title: {row['title']}\n"
        f"DOI: {row['doi']}\n"
        f"Publication Date: {row['publication_date']}\n"
        f"Cited By: {row['cited_by_count']}\n"
        f"Authors: {row['authors']}\n"
        f"Journal: {row['journal']}\n"
        f"Open Access: {row['open_access']}\n"
        f"Type: {row['type']}\n"
        f"Topic: {row['topic']}\n"
        "---\n"
    )
    chunks.append(chunk)

print(f"📊 Created {len(chunks)} chunks.\n")

# ================================================================
# 3. LOAD LOCAL EMBEDDING MODEL (free, no API)
# ================================================================

print("⏳ Loading embedding model (all-MiniLM-L6-v2)...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')
print("✅ Model loaded.\n")

# ================================================================
# 4. BUILD FAISS INDEX (with checkpoint)
# ================================================================

INDEX_FILE = "faiss_index.pkl"
CHUNKS_FILE = "chunks.pkl"

if os.path.exists(INDEX_FILE) and os.path.exists(CHUNKS_FILE):
    print("🔄 Loading saved index and chunks...")
    with open(CHUNKS_FILE, 'rb') as f:
        chunks = pickle.load(f)
    with open(INDEX_FILE, 'rb') as f:
        index = pickle.load(f)
    print("✅ Loaded FAISS index.\n")
else:
    print("⏳ Generating embeddings...")
    embeddings = embedder.encode(chunks, convert_to_numpy=True, show_progress_bar=True)
    print(f"✅ Generated {len(embeddings)} embeddings, dimension={embeddings.shape[1]}")

    dim = embeddings.shape[1]
    index = faiss.IndexFlatL2(dim)
    index.add(embeddings.astype('float32'))
    print("✅ FAISS index built.")

    with open(INDEX_FILE, 'wb') as f:
        pickle.dump(index, f)
    with open(CHUNKS_FILE, 'wb') as f:
        pickle.dump(chunks, f)
    print("💾 Index and chunks saved.\n")

# ================================================================
# 5. GET API KEYS (from Colab Secrets or prompt)
# ================================================================

def get_key(name, prompt):
    if IN_COLAB:
        try:
            key = userdata.get(name)
            if key:
                print(f"✅ Loaded {name} from Colab Secrets.")
                return key
        except:
            pass
    key = input(prompt).strip()
    if not key:
        raise ValueError("❌ API key cannot be empty.")
    return key

# ================================================================
# 6. SET UP GENERATORS (Gemini, DeepSeek, or local fallback)
# ================================================================

USE_API_LLMS = True   # set to False to use local model only

if USE_API_LLMS:
    try:
        GEMINI_API_KEY = get_key("GEMINI_API_KEY", "Enter Gemini API Key: ")
        DEEPSEEK_API_KEY = get_key("DEEPSEEK_API_KEY", "Enter DeepSeek API Key: ")
        genai.configure(api_key=GEMINI_API_KEY)
        deepseek_client = OpenAI(
            api_key=DEEPSEEK_API_KEY,
            base_url="https://api.deepseek.com/v1"
        )
        print("✅ API keys configured.\n")
        api_ok = True
    except Exception as e:
        print(f"⚠️ API key setup failed: {e}. Falling back to local model.")
        USE_API_LLMS = False
        api_ok = False

# ---------- Gemini (with corrected model name) ----------
def ask_gemini(context, question):
    prompt = f"""Answer based ONLY on the context below. If not found, say "I cannot find this in the provided text."

--- CONTEXT ---
{context}
--- END CONTEXT ---

Question: {question}
Answer:"""
    try:
        # Use a model that exists in the current API
        model = genai.GenerativeModel('gemini-1.5-flash')   # <-- FIXED
        response = model.generate_content(prompt)
        return response.text.strip()
    except Exception as e:
        return f"❌ Gemini error: {e}"

# ---------- DeepSeek ----------
def ask_deepseek(context, question):
    prompt = f"""Answer based ONLY on the context below. If not found, say "I cannot find this in the provided text."

--- CONTEXT ---
{context}
--- END CONTEXT ---

Question: {question}
Answer:"""
    try:
        response = deepseek_client.chat.completions.create(
            model="deepseek-chat",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.3,
            max_tokens=300
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"❌ DeepSeek error: {e}"

# ---------- Local fallback (no API) ----------
from transformers import pipeline

def ask_local(context, question):
    prompt = f"Answer based on context: {context}\nQuestion: {question}\nAnswer:"
    try:
        # Load the model once and cache it
        if not hasattr(ask_local, "generator"):
            ask_local.generator = pipeline("text2text-generation", model="google/flan-t5-small")
        result = ask_local.generator(prompt, max_length=150, do_sample=False)[0]['generated_text']
        return result
    except Exception as e:
        return f"❌ Local model error: {e}"

# ---------- Choose the generator ----------
if USE_API_LLMS and api_ok:
    def answer_question(context, question):
        # You can switch between Gemini and DeepSeek here
        # For now we try Gemini, fallback to DeepSeek, then local
        ans = ask_gemini(context, question)
        if ans.startswith("❌ Gemini error"):
            ans = ask_deepseek(context, question)
            if ans.startswith("❌ DeepSeek error"):
                ans = ask_local(context, question)
        return ans
else:
    def answer_question(context, question):
        return ask_local(context, question)

# ================================================================
# 7. RETRIEVAL FUNCTION
# ================================================================

def retrieve(query, top_k=3):
    query_emb = embedder.encode([query], convert_to_numpy=True)[0]
    distances, indices = index.search(np.array([query_emb]).astype('float32'), top_k)
    results = []
    for i, idx in enumerate(indices[0]):
        results.append({
            'chunk': chunks[idx],
            'score': float(distances[0][i])
        })
    return results

# ================================================================
# 8. INTERACTIVE Q&A LOOP
# ================================================================

print("="*60)
print("💬 Ask questions about the papers (type 'exit' to quit)")
print("Example: 'What are the most cited papers on Machine Learning?'")
print("="*60)

while True:
    q = input("\n❓ Your question: ").strip()
    if q.lower() in ['exit', 'quit', 'q']:
        print("👋 Goodbye!")
        break
    if not q:
        continue

    print("🔍 Searching...")
    results = retrieve(q, top_k=3)
    if not results:
        print("❌ No results.")
        continue

    context = "\n\n---\n\n".join([r['chunk'] for r in results])
    print(f"✅ Retrieved {len(results)} papers.\n")

    print("-"*60)
    print("📚 RETRIEVED CONTEXT (first 500 chars):")
    print("-"*60)
    print(textwrap.fill(context[:500], width=70))
    print("\n" + "-"*60)

    print("🤖 ANSWER:")
    print("-"*60)
    answer = answer_question(context, q)
    print(textwrap.fill(answer, width=70))
    print("\n" + "-"*60)

📤 Upload your CSV file (or press Enter to use the embedded sample)


Saving AI_ML_Research_Papers_2023_2026.csv to AI_ML_Research_Papers_2023_2026 (3).csv
✅ Loaded 704 rows from AI_ML_Research_Papers_2023_2026 (3).csv
📋 Columns: ['title', 'doi', 'publication_date', 'cited_by_count', 'authors', 'journal', 'open_access', 'type', 'topic']
📊 Created 704 chunks.

⏳ Loading embedding model (all-MiniLM-L6-v2)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Model loaded.

🔄 Loading saved index and chunks...
✅ Loaded FAISS index.

Enter Gemini API Key: AQ.Ab8RN6LvRi93jDDvurGEZFsfks4MkEOc-VfvilNVo7qzLKEb0w
✅ Loaded DEEPSEEK_API_KEY from Colab Secrets.
✅ API keys configured.

💬 Ask questions about the papers (type 'exit' to quit)
Example: 'What are the most cited papers on Machine Learning?'

❓ Your question: Summarize AlphaFold3
🔍 Searching...
✅ Retrieved 3 papers.

------------------------------------------------------------
📚 RETRIEVED CONTEXT (first 500 chars):
------------------------------------------------------------
Title: AlphaFold Protein Structure Database in 2024: providing
structure coverage for over 214 million protein sequences DOI:
https://doi.org/10.1093/nar/gkad1011 Publication Date: 2023-11-02
Cited By: 1737 Authors: Mihály Váradi, Damian Bertoni, Paulyna Magaña,
Urmila Paramval, Ivanna Pidruchna Journal: Nucleic Acids Research Open
Access: True Type: article Topic: Machine Learning ---   ---  Title:
Accurate structure 

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

❌ Local model error: "Unknown task text2text-generation, available
tasks are ['any-to-any', 'audio-classification', 'automatic-speech-
recognition', 'depth-estimation', 'document-question-answering',
'feature-extraction', 'fill-mask', 'image-classification', 'image-
feature-extraction', 'image-segmentation', 'image-text-to-text',
'keypoint-matching', 'mask-generation', 'ner', 'object-detection',
'sentiment-analysis', 'table-question-answering', 'text-
classification', 'text-generation', 'text-to-audio', 'text-to-speech',
'token-classification', 'video-classification', 'zero-shot-audio-
classification', 'zero-shot-classification', 'zero-shot-image-
classification', 'zero-shot-object-detection']"

------------------------------------------------------------

❓ Your question: Tell me about the paper 'MizAR 60 for Mizar 50'.
🔍 Searching...
✅ Retrieved 3 papers.

------------------------------------------------------------
📚 RETRIEVED CONTEXT (first 500 chars):
-----------------------------

❌ Local model error: "Unknown task text2text-generation, available
tasks are ['any-to-any', 'audio-classification', 'automatic-speech-
recognition', 'depth-estimation', 'document-question-answering',
'feature-extraction', 'fill-mask', 'image-classification', 'image-
feature-extraction', 'image-segmentation', 'image-text-to-text',
'keypoint-matching', 'mask-generation', 'ner', 'object-detection',
'sentiment-analysis', 'table-question-answering', 'text-
classification', 'text-generation', 'text-to-audio', 'text-to-speech',
'token-classification', 'video-classification', 'zero-shot-audio-
classification', 'zero-shot-classification', 'zero-shot-image-
classification', 'zero-shot-object-detection']"

------------------------------------------------------------

❓ Your question: What is the paper 'LLaMA
🔍 Searching...
✅ Retrieved 3 papers.

------------------------------------------------------------
📚 RETRIEVED CONTEXT (first 500 chars):
-----------------------------------------------------

❌ Local model error: "Unknown task text2text-generation, available
tasks are ['any-to-any', 'audio-classification', 'automatic-speech-
recognition', 'depth-estimation', 'document-question-answering',
'feature-extraction', 'fill-mask', 'image-classification', 'image-
feature-extraction', 'image-segmentation', 'image-text-to-text',
'keypoint-matching', 'mask-generation', 'ner', 'object-detection',
'sentiment-analysis', 'table-question-answering', 'text-
classification', 'text-generation', 'text-to-audio', 'text-to-speech',
'token-classification', 'video-classification', 'zero-shot-audio-
classification', 'zero-shot-classification', 'zero-shot-image-
classification', 'zero-shot-object-detection']"

------------------------------------------------------------

❓ Your question: exit
👋 Goodbye!


In [7]:
# ================================================================
# RAG on Academic Papers Dataset (AI/ML Research Papers 2023-2026)
# ================================================================

import os
import pickle
import textwrap
import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer
import google.generativeai as genai
from openai import OpenAI
import io
from transformers import T5Tokenizer, T5ForConditionalGeneration
import torch

# ---------- Check if running in Colab ----------
try:
    from google.colab import files, userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# ---------- Install required packages ----------
!pip install -q sentence-transformers faiss-cpu openai google-generativeai transformers torch

# ================================================================
# 1. GET THE CSV FILE (upload or use embedded sample)
# ================================================================

print("📤 Upload your CSV file (or press Enter to use the embedded sample)")
try:
    uploaded = files.upload()
    if uploaded:
        filename = list(uploaded.keys())[0]
        content = uploaded[filename].decode('utf-8-sig')
        df = pd.read_csv(io.StringIO(content))
        print(f"✅ Loaded {len(df)} rows from {filename}")
    else:
        raise Exception("No file uploaded")
except:
    print("⚠️ No file uploaded. Using a small sample dataset for demo.")
    sample_data = {
        'title': ['MizAR 60 for Mizar 50', 'A Multi-Modal Distributed Real-Time IoT System', 'On a Method to Measure Supervised Multiclass Model’s Interpretability'],
        'doi': ['https://doi.org/10.4230/lipics.itp.2023.19', 'https://doi.org/10.4230/oasics.ng-res.2024.2', 'https://doi.org/10.4230/oasics.dx.2024.27'],
        'publication_date': ['2023-01-01', '2024-01-01', '2024-01-01'],
        'cited_by_count': [74707, 14276, 13164],
        'authors': ['Jakubův, Jan, Chvalovský, Karel, Goertzel, Zarathustra, Kaliszyk, Cezary, Olšák, Mirek', 'Khanam, Zeba, Achari, Vejey Pradeep Suresh, Boukhennoufa, Issam, Jindal, Anish, Singh, Amit Kumar', 'Gauriat, Charles-Maxime, Pencolé, Yannick, Ribot, Pauline, Brouillet, Gregory'],
        'journal': ['DROPS (Schloss Dagstuhl – Leibniz Center for Informatics)', 'DROPS (Schloss Dagstuhl – Leibniz Center for Informatics)', 'Dagstuhl Research Online Publication Server'],
        'open_access': [True, True, True],
        'type': ['preprint', 'preprint', 'preprint'],
        'topic': ['Machine Learning', 'Machine Learning', 'Machine Learning']
    }
    df = pd.DataFrame(sample_data)
    print(f"✅ Using sample of {len(df)} papers")

df.columns = df.columns.str.strip()
print("📋 Columns:", list(df.columns))

# ================================================================
# 2. CREATE CHUNKS (one per paper)
# ================================================================

chunks = []
for _, row in df.iterrows():
    chunk = (
        f"Title: {row['title']}\n"
        f"DOI: {row['doi']}\n"
        f"Publication Date: {row['publication_date']}\n"
        f"Cited By: {row['cited_by_count']}\n"
        f"Authors: {row['authors']}\n"
        f"Journal: {row['journal']}\n"
        f"Open Access: {row['open_access']}\n"
        f"Type: {row['type']}\n"
        f"Topic: {row['topic']}\n"
        "---\n"
    )
    chunks.append(chunk)

print(f"📊 Created {len(chunks)} chunks.\n")

# ================================================================
# 3. LOAD LOCAL EMBEDDING MODEL (free, no API)
# ================================================================

print("⏳ Loading embedding model (all-MiniLM-L6-v2)...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')
print("✅ Model loaded.\n")

# ================================================================
# 4. BUILD FAISS INDEX (with checkpoint)
# ================================================================

INDEX_FILE = "faiss_index.pkl"
CHUNKS_FILE = "chunks.pkl"

if os.path.exists(INDEX_FILE) and os.path.exists(CHUNKS_FILE):
    print("🔄 Loading saved index and chunks...")
    with open(CHUNKS_FILE, 'rb') as f:
        chunks = pickle.load(f)
    with open(INDEX_FILE, 'rb') as f:
        index = pickle.load(f)
    print("✅ Loaded FAISS index.\n")
else:
    print("⏳ Generating embeddings...")
    embeddings = embedder.encode(chunks, convert_to_numpy=True, show_progress_bar=True)
    print(f"✅ Generated {len(embeddings)} embeddings, dimension={embeddings.shape[1]}")

    dim = embeddings.shape[1]
    index = faiss.IndexFlatL2(dim)
    index.add(embeddings.astype('float32'))
    print("✅ FAISS index built.")

    with open(INDEX_FILE, 'wb') as f:
        pickle.dump(index, f)
    with open(CHUNKS_FILE, 'wb') as f:
        pickle.dump(chunks, f)
    print("💾 Index and chunks saved.\n")

# ================================================================
# 5. GET API KEYS (from Colab Secrets or prompt)
# ================================================================

def get_key(name, prompt):
    if IN_COLAB:
        try:
            key = userdata.get(name)
            if key:
                print(f"✅ Loaded {name} from Colab Secrets.")
                return key
        except:
            pass
    key = input(prompt).strip()
    if not key:
        raise ValueError("❌ API key cannot be empty.")
    return key

# ================================================================
# 6. SET UP GENERATORS (Gemini, DeepSeek, or local fallback)
# ================================================================

USE_API_LLMS = True   # set to False to use local model only

if USE_API_LLMS:
    try:
        GEMINI_API_KEY = get_key("GEMINI_API_KEY", "Enter Gemini API Key: ")
        DEEPSEEK_API_KEY = get_key("DEEPSEEK_API_KEY", "Enter DeepSeek API Key: ")
        genai.configure(api_key=GEMINI_API_KEY)
        deepseek_client = OpenAI(
            api_key=DEEPSEEK_API_KEY,
            base_url="https://api.deepseek.com/v1"
        )
        print("✅ API keys configured.\n")
        api_ok = True
    except Exception as e:
        print(f"⚠️ API key setup failed: {e}. Falling back to local model.")
        USE_API_LLMS = False
        api_ok = False

# ---------- Gemini (use 'gemini-pro' – always available) ----------
def ask_gemini(context, question):
    prompt = f"""Answer based ONLY on the context below. If not found, say "I cannot find this in the provided text."

--- CONTEXT ---
{context}
--- END CONTEXT ---

Question: {question}
Answer:"""
    try:
        model = genai.GenerativeModel('gemini-pro')   # <-- reliable model
        response = model.generate_content(prompt)
        return response.text.strip()
    except Exception as e:
        return f"❌ Gemini error: {e}"

# ---------- DeepSeek ----------
def ask_deepseek(context, question):
    prompt = f"""Answer based ONLY on the context below. If not found, say "I cannot find this in the provided text."

--- CONTEXT ---
{context}
--- END CONTEXT ---

Question: {question}
Answer:"""
    try:
        response = deepseek_client.chat.completions.create(
            model="deepseek-chat",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.3,
            max_tokens=300
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"❌ DeepSeek error: {e}"

# ---------- Local fallback (no pipeline, direct model loading) ----------
def ask_local(context, question):
    # Cache the model and tokenizer
    if not hasattr(ask_local, "tokenizer"):
        print("⏳ Loading local T5 model (this may take a moment)...")
        ask_local.tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-small")
        ask_local.model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-small")
        ask_local.model.eval()
        print("✅ Local model ready.")

    prompt = f"Answer based on context: {context}\nQuestion: {question}\nAnswer:"
    inputs = ask_local.tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        outputs = ask_local.model.generate(**inputs, max_new_tokens=150, do_sample=False)
    answer = ask_local.tokenizer.decode(outputs[0], skip_special_tokens=True)
    return answer if answer.strip() else "I cannot find this in the provided text."

# ---------- Choose the generator ----------
if USE_API_LLMS and api_ok:
    def answer_question(context, question):
        ans = ask_gemini(context, question)
        if ans.startswith("❌ Gemini error"):
            ans = ask_deepseek(context, question)
            if ans.startswith("❌ DeepSeek error"):
                ans = ask_local(context, question)
        return ans
else:
    def answer_question(context, question):
        return ask_local(context, question)

# ================================================================
# 7. RETRIEVAL FUNCTION
# ================================================================

def retrieve(query, top_k=3):
    query_emb = embedder.encode([query], convert_to_numpy=True)[0]
    distances, indices = index.search(np.array([query_emb]).astype('float32'), top_k)
    results = []
    for i, idx in enumerate(indices[0]):
        results.append({
            'chunk': chunks[idx],
            'score': float(distances[0][i])
        })
    return results

# ================================================================
# 8. INTERACTIVE Q&A LOOP
# ================================================================

print("="*60)
print("💬 Ask questions about the papers (type 'exit' to quit)")
print("Example: 'What are the most cited papers on Machine Learning?'")
print("="*60)

while True:
    q = input("\n❓ Your question: ").strip()
    if q.lower() in ['exit', 'quit', 'q']:
        print("👋 Goodbye!")
        break
    if not q:
        continue

    print("🔍 Searching...")
    results = retrieve(q, top_k=3)
    if not results:
        print("❌ No results.")
        continue

    context = "\n\n---\n\n".join([r['chunk'] for r in results])
    print(f"✅ Retrieved {len(results)} papers.\n")

    print("-"*60)
    print("📚 RETRIEVED CONTEXT (first 500 chars):")
    print("-"*60)
    print(textwrap.fill(context[:500], width=70))
    print("\n" + "-"*60)

    print("🤖 ANSWER:")
    print("-"*60)
    answer = answer_question(context, q)
    print(textwrap.fill(answer, width=70))
    print("\n" + "-"*60)

📤 Upload your CSV file (or press Enter to use the embedded sample)


Saving AI_ML_Research_Papers_2023_2026.csv to AI_ML_Research_Papers_2023_2026 (4).csv
✅ Loaded 704 rows from AI_ML_Research_Papers_2023_2026 (4).csv
📋 Columns: ['title', 'doi', 'publication_date', 'cited_by_count', 'authors', 'journal', 'open_access', 'type', 'topic']
📊 Created 704 chunks.

⏳ Loading embedding model (all-MiniLM-L6-v2)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Model loaded.

🔄 Loading saved index and chunks...
✅ Loaded FAISS index.

Enter Gemini API Key: AQ.Ab8RN6LvRi93jDDvurGEZFsfks4MkEOc-VfvilNVo7qzLKEb0w
✅ Loaded DEEPSEEK_API_KEY from Colab Secrets.
✅ API keys configured.

💬 Ask questions about the papers (type 'exit' to quit)
Example: 'What are the most cited papers on Machine Learning?'

❓ Your question: Summarize the paper 'MizAR 60 for Mizar 50'.
🔍 Searching...
✅ Retrieved 3 papers.

------------------------------------------------------------
📚 RETRIEVED CONTEXT (first 500 chars):
------------------------------------------------------------
Title: MizAR 60 for Mizar 50 DOI:
https://doi.org/10.4230/lipics.itp.2023.19 Publication Date:
2023-01-01 Cited By: 74707 Authors: Jakubův, Jan, Chvalovský, Karel,
Goertzel, Zarathustra, Kaliszyk, Cezary, Olšák, Mirek Journal: DROPS
(Schloss Dagstuhl – Leibniz Center for Informatics) Open Access: True
Type: preprint Topic: Machine Learning ---   ---  Title: Prisoner’s
dilemma game model Based on

⏳ Loading local T5 model (this may take a moment)...


...

LLMs

------------------------------------------------------------

❓ Your question: List the top 3 most cited papers about Machine Learning
🔍 Searching...
✅ Retrieved 3 papers.

------------------------------------------------------------
📚 RETRIEVED CONTEXT (first 500 chars):
------------------------------------------------------------
Title: Evaluation metrics and statistical tests for machine learning
DOI: https://doi.org/10.1038/s41598-024-56706-x Publication Date:
2024-03-13 Cited By: 885 Authors: Oona Rainio, Jarmo Teuho, Riku Klén
Journal: Scientific Reports Open Access: True Type: article Topic:
Machine Learning ---   ---  Title: The school and society DOI:
https://doi.org/10.55094/holistence.497 Publication Date: 2024-01-12
Cited By: 1489 Authors: John Dewey Journal: nan Open Access: True
Type: book Topic: Machine Lear

------------------------------------------------------------
🤖 ANSWER:
------------------------------------------------------------


List the top 3 papers about machine learning

------------------------------------------------------------

❓ Your question: What is the average citation count among papers published in Nature?
🔍 Searching...
✅ Retrieved 3 papers.

------------------------------------------------------------
📚 RETRIEVED CONTEXT (first 500 chars):
------------------------------------------------------------
Title: Papers and patents are becoming less disruptive over time DOI:
https://doi.org/10.1038/s41586-022-05543-x Publication Date:
2023-01-04 Cited By: 753 Authors: Michael Park, Erin Leahey, Russell
J. Funk Journal: Nature Open Access: True Type: article Topic: LLMs
---   ---  Title: How to design bibliometric research: an overview and
a framework proposal DOI: https://doi.org/10.1007/s11846-024-00738-0
Publication Date: 2024-03-06 Cited By: 516 Authors: Oğuzhan Öztürk,
Rıdvan Kocaman, Dominik

------------------------------------------------------------
🤖 ANSWER:
-----------------------------------

citation count is .

------------------------------------------------------------

❓ Your question: Which papers have 'Yoshua Bengio' as an author?
🔍 Searching...
✅ Retrieved 3 papers.

------------------------------------------------------------
📚 RETRIEVED CONTEXT (first 500 chars):
------------------------------------------------------------
Title: A Review on YOLOv8 and Its Advancements DOI:
https://doi.org/10.1007/978-981-99-7962-2_39 Publication Date:
2024-01-01 Cited By: 625 Authors: Mupparaju Sohan, Thotakura Sai Ram,
Ch. Venkata Rami Reddy Journal: Algorithms for intelligent systems
Open Access: False Type: review Topic: Machine Learning ---   ---
Title: Network pharmacology: towards the artificial intelligence-based
precision traditional Chinese medicine DOI:
https://doi.org/10.1093/bib/bbad518 Publication Date: 2023-11-22 C

------------------------------------------------------------
🤖 ANSWER:
------------------------------------------------------------


AI Agents

------------------------------------------------------------

❓ Your question: List the authors of the paper on 'ChatGPT for good?
🔍 Searching...
✅ Retrieved 3 papers.

------------------------------------------------------------
📚 RETRIEVED CONTEXT (first 500 chars):
------------------------------------------------------------
Title: Fabrication and errors in the bibliographic citations generated
by ChatGPT DOI: https://doi.org/10.1038/s41598-023-41032-5 Publication
Date: 2023-09-07 Cited By: 301 Authors: William H. Walters, Esther
Isabelle Wilder Journal: Scientific Reports Open Access: True Type:
article Topic: RAG ---   ---  Title: A Brief Overview of ChatGPT: The
History, Status Quo and Potential Future Development DOI:
https://doi.org/10.1109/jas.2023.123618 Publication Date: 2023-05-01
Cited By: 1341 Authors: Ti

------------------------------------------------------------
🤖 ANSWER:
------------------------------------------------------------


ChatGPT

------------------------------------------------------------

❓ Your question: exit
👋 Goodbye!


In [8]:
# ================================================================
# RAG on Academic Papers (DeepSeek + Local Fallback)
# ================================================================

import os
import pickle
import textwrap
import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer
from openai import OpenAI
import io
from transformers import T5Tokenizer, T5ForConditionalGeneration
import torch

# ---------- Check if running in Colab ----------
try:
    from google.colab import files, userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# ---------- Install required packages ----------
!pip install -q sentence-transformers faiss-cpu openai transformers torch

# ================================================================
# 1. GET THE CSV FILE (upload or use sample)
# ================================================================

print("📤 Upload your CSV file (or press Enter to use the embedded sample)")
try:
    uploaded = files.upload()
    if uploaded:
        filename = list(uploaded.keys())[0]
        content = uploaded[filename].decode('utf-8-sig')
        df = pd.read_csv(io.StringIO(content))
        print(f"✅ Loaded {len(df)} rows from {filename}")
    else:
        raise Exception("No file uploaded")
except:
    print("⚠️ No file uploaded. Using a small sample dataset for demo.")
    sample_data = {
        'title': ['MizAR 60 for Mizar 50', 'A Multi-Modal Distributed Real-Time IoT System', 'On a Method to Measure Supervised Multiclass Model’s Interpretability'],
        'doi': ['https://doi.org/10.4230/lipics.itp.2023.19', 'https://doi.org/10.4230/oasics.ng-res.2024.2', 'https://doi.org/10.4230/oasics.dx.2024.27'],
        'publication_date': ['2023-01-01', '2024-01-01', '2024-01-01'],
        'cited_by_count': [74707, 14276, 13164],
        'authors': ['Jakubův, Jan, Chvalovský, Karel, Goertzel, Zarathustra, Kaliszyk, Cezary, Olšák, Mirek', 'Khanam, Zeba, Achari, Vejey Pradeep Suresh, Boukhennoufa, Issam, Jindal, Anish, Singh, Amit Kumar', 'Gauriat, Charles-Maxime, Pencolé, Yannick, Ribot, Pauline, Brouillet, Gregory'],
        'journal': ['DROPS (Schloss Dagstuhl – Leibniz Center for Informatics)', 'DROPS (Schloss Dagstuhl – Leibniz Center for Informatics)', 'Dagstuhl Research Online Publication Server'],
        'open_access': [True, True, True],
        'type': ['preprint', 'preprint', 'preprint'],
        'topic': ['Machine Learning', 'Machine Learning', 'Machine Learning']
    }
    df = pd.DataFrame(sample_data)
    print(f"✅ Using sample of {len(df)} papers")

df.columns = df.columns.str.strip()
print("📋 Columns:", list(df.columns))

# ================================================================
# 2. CREATE CHUNKS (one per paper)
# ================================================================

chunks = []
for _, row in df.iterrows():
    chunk = (
        f"Title: {row['title']}\n"
        f"DOI: {row['doi']}\n"
        f"Publication Date: {row['publication_date']}\n"
        f"Cited By: {row['cited_by_count']}\n"
        f"Authors: {row['authors']}\n"
        f"Journal: {row['journal']}\n"
        f"Open Access: {row['open_access']}\n"
        f"Type: {row['type']}\n"
        f"Topic: {row['topic']}\n"
        "---\n"
    )
    chunks.append(chunk)

print(f"📊 Created {len(chunks)} chunks.\n")

# ================================================================
# 3. LOAD LOCAL EMBEDDING MODEL (free, no API)
# ================================================================

print("⏳ Loading embedding model (all-MiniLM-L6-v2)...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')
print("✅ Model loaded.\n")

# ================================================================
# 4. BUILD FAISS INDEX (with checkpoint)
# ================================================================

INDEX_FILE = "faiss_index.pkl"
CHUNKS_FILE = "chunks.pkl"

if os.path.exists(INDEX_FILE) and os.path.exists(CHUNKS_FILE):
    print("🔄 Loading saved index and chunks...")
    with open(CHUNKS_FILE, 'rb') as f:
        chunks = pickle.load(f)
    with open(INDEX_FILE, 'rb') as f:
        index = pickle.load(f)
    print("✅ Loaded FAISS index.\n")
else:
    print("⏳ Generating embeddings...")
    embeddings = embedder.encode(chunks, convert_to_numpy=True, show_progress_bar=True)
    print(f"✅ Generated {len(embeddings)} embeddings, dimension={embeddings.shape[1]}")

    dim = embeddings.shape[1]
    index = faiss.IndexFlatL2(dim)
    index.add(embeddings.astype('float32'))
    print("✅ FAISS index built.")

    with open(INDEX_FILE, 'wb') as f:
        pickle.dump(index, f)
    with open(CHUNKS_FILE, 'wb') as f:
        pickle.dump(chunks, f)
    print("💾 Index and chunks saved.\n")

# ================================================================
# 5. GET DEEPSEEK API KEY
# ================================================================

def get_key(name, prompt):
    if IN_COLAB:
        try:
            key = userdata.get(name)
            if key:
                print(f"✅ Loaded {name} from Colab Secrets.")
                return key
        except:
            pass
    key = input(prompt).strip()
    if not key:
        raise ValueError("❌ API key cannot be empty.")
    return key

# ================================================================
# 6. SET UP GENERATORS (DeepSeek + Local fallback)
# ================================================================

USE_DEEPSEEK = True   # set to False to use local only

if USE_DEEPSEEK:
    try:
        DEEPSEEK_API_KEY = get_key("DEEPSEEK_API_KEY", "Enter DeepSeek API Key: ")
        deepseek_client = OpenAI(
            api_key=DEEPSEEK_API_KEY,
            base_url="https://api.deepseek.com/v1"
        )
        print("✅ DeepSeek API key configured.\n")
        deepseek_ok = True
    except Exception as e:
        print(f"⚠️ DeepSeek setup failed: {e}. Falling back to local model.")
        USE_DEEPSEEK = False
        deepseek_ok = False

# ---------- DeepSeek ----------
def ask_deepseek(context, question):
    prompt = f"""Answer based ONLY on the context below. If not found, say "I cannot find this in the provided text."

--- CONTEXT ---
{context}
--- END CONTEXT ---

Question: {question}
Answer:"""
    try:
        response = deepseek_client.chat.completions.create(
            model="deepseek-chat",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.3,
            max_tokens=300
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"❌ DeepSeek error: {e}"

# ---------- Local fallback (using a larger model) ----------
def ask_local(context, question):
    # Cache the model and tokenizer
    if not hasattr(ask_local, "tokenizer"):
        print("⏳ Loading local T5 model (this may take a moment)...")
        # Use a larger model for better quality
        ask_local.tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-base")
        ask_local.model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-base")
        ask_local.model.eval()
        print("✅ Local model ready.")

    prompt = f"Answer based on context: {context}\nQuestion: {question}\nAnswer:"
    inputs = ask_local.tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        outputs = ask_local.model.generate(**inputs, max_new_tokens=150, do_sample=False)
    answer = ask_local.tokenizer.decode(outputs[0], skip_special_tokens=True)
    return answer if answer.strip() else "I cannot find this in the provided text."

# ---------- Choose the generator ----------
if USE_DEEPSEEK and deepseek_ok:
    def answer_question(context, question):
        ans = ask_deepseek(context, question)
        if ans.startswith("❌ DeepSeek error"):
            ans = ask_local(context, question)
        return ans
else:
    def answer_question(context, question):
        return ask_local(context, question)

# ================================================================
# 7. RETRIEVAL FUNCTION
# ================================================================

def retrieve(query, top_k=3):
    query_emb = embedder.encode([query], convert_to_numpy=True)[0]
    distances, indices = index.search(np.array([query_emb]).astype('float32'), top_k)
    results = []
    for i, idx in enumerate(indices[0]):
        results.append({
            'chunk': chunks[idx],
            'score': float(distances[0][i])
        })
    return results

# ================================================================
# 8. INTERACTIVE Q&A LOOP
# ================================================================

print("="*60)
print("💬 Ask questions about the papers (type 'exit' to quit)")
print("Example: 'What are the most cited papers on Machine Learning?'")
print("="*60)

while True:
    q = input("\n❓ Your question: ").strip()
    if q.lower() in ['exit', 'quit', 'q']:
        print("👋 Goodbye!")
        break
    if not q:
        continue

    print("🔍 Searching...")
    results = retrieve(q, top_k=3)
    if not results:
        print("❌ No results.")
        continue

    context = "\n\n---\n\n".join([r['chunk'] for r in results])
    print(f"✅ Retrieved {len(results)} papers.\n")

    print("-"*60)
    print("📚 RETRIEVED CONTEXT (first 500 chars):")
    print("-"*60)
    print(textwrap.fill(context[:500], width=70))
    print("\n" + "-"*60)

    print("🤖 ANSWER:")
    print("-"*60)
    answer = answer_question(context, q)
    print(textwrap.fill(answer, width=70))
    print("\n" + "-"*60)

📤 Upload your CSV file (or press Enter to use the embedded sample)


Saving AI_ML_Research_Papers_2023_2026.csv to AI_ML_Research_Papers_2023_2026 (5).csv
✅ Loaded 704 rows from AI_ML_Research_Papers_2023_2026 (5).csv
📋 Columns: ['title', 'doi', 'publication_date', 'cited_by_count', 'authors', 'journal', 'open_access', 'type', 'topic']
📊 Created 704 chunks.

⏳ Loading embedding model (all-MiniLM-L6-v2)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Model loaded.

🔄 Loading saved index and chunks...
✅ Loaded FAISS index.

✅ Loaded DEEPSEEK_API_KEY from Colab Secrets.
✅ DeepSeek API key configured.

💬 Ask questions about the papers (type 'exit' to quit)
Example: 'What are the most cited papers on Machine Learning?'

❓ Your question: Most cited papers on LLM
🔍 Searching...
✅ Retrieved 3 papers.

------------------------------------------------------------
📚 RETRIEVED CONTEXT (first 500 chars):
------------------------------------------------------------
Title: Papers and patents are becoming less disruptive over time DOI:
https://doi.org/10.1038/s41586-022-05543-x Publication Date:
2023-01-04 Cited By: 753 Authors: Michael Park, Erin Leahey, Russell
J. Funk Journal: Nature Open Access: True Type: article Topic: LLMs
---   ---  Title: Harnessing the Power of LLMs in Practice: A Survey
on ChatGPT and Beyond DOI: https://doi.org/10.1145/3649506 Publication
Date: 2024-02-28 Cited By: 442 Authors: Jingfeng Yang, Hongye Jin,
Ruixiang Ta

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

✅ Local model ready.
--- ---

------------------------------------------------------------

❓ Your question: Show me papers by Som Biswas.
🔍 Searching...
✅ Retrieved 3 papers.

------------------------------------------------------------
📚 RETRIEVED CONTEXT (first 500 chars):
------------------------------------------------------------
Title: Prompt Engineering with ChatGPT: A Guide for Academic Writers
DOI: https://doi.org/10.1007/s10439-023-03272-4 Publication Date:
2023-06-07 Cited By: 644 Authors: Louie Giray Journal: Annals of
Biomedical Engineering Open Access: False Type: letter Topic: LLMs ---
---  Title: Papers and patents are becoming less disruptive over time
DOI: https://doi.org/10.1038/s41586-022-05543-x Publication Date:
2023-01-04 Cited By: 753 Authors: Michael Park, Erin Leahey, Russell
J. Funk Journal: Natur

------------------------------------------------------------
🤖 ANSWER:
------------------------------------------------------------
I cannot find this in the prov